####Transform Orders Data
- Access elements from JSON object
- Deduplicate Array Elements
- Explode Arrays
- Write transformed data to silver layer

#### 1. Access elements from JSON object

In [0]:
 from pyspark.sql import functions as F

df = spark.read.table("gizmobox.silver.py_orders_json")
mapped_df = df.select(
    "json_value.customer_id",
    "json_value.order_date",
    "json_value.order_id",
    "json_value.order_status",
    "json_value.payment_method",
    "json_value.total_amount",
    "json_value.transaction_timestamp",
    "json_value.items"
)
display(mapped_df)

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("gizmobox.silver.py_orders_json")
mapped_df = df.select(
    df.json_value.customer_id.alias("customer_id"),
    df.json_value.order_date.alias("order_date"),
    df.json_value.order_id.alias("order_id"),
    df.json_value.order_status.alias("order_status"),
    df.json_value.payment_method.alias("payment_method"),
    df.json_value.total_amount.alias("total_amount"),
    df.json_value.transaction_timestamp.alias("transaction_timestamp"),
    df.json_value.items.alias("items")
    )
display(mapped_df)

####2. Deduplicate Array Elements
- https://docs.databricks.com/aws/en/sql/language-manual/functions/array_distinct

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("gizmobox.silver.py_orders_json")
mapped_df = df.select(
    df.json_value.customer_id.alias("customer_id"),
    df.json_value.order_date.alias("order_date"),
    df.json_value.order_id.alias("order_id"),
    df.json_value.order_status.alias("order_status"),
    df.json_value.payment_method.alias("payment_method"),
    df.json_value.total_amount.alias("total_amount"),
    df.json_value.transaction_timestamp.alias("transaction_timestamp"),
    F.array_distinct(df.json_value.items).alias("items")
    )
display(mapped_df)

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("gizmobox.silver.py_orders_json")
mapped_df = df.select(
    "json_value.customer_id",
    "json_value.order_date",
    "json_value.order_id",
    "json_value.order_status",
    "json_value.payment_method",
    "json_value.total_amount",
    "json_value.transaction_timestamp",
     F.array_distinct("json_value.items").alias("items")
    )
display(mapped_df)

####3. Explode Arrays
- https://docs.databricks.com/aws/en/pyspark/reference/functions/explode

In [0]:
from pyspark.sql import functions as F

exploded_df = mapped_df.select(
    "customer_id",
    "order_date",
    "order_id",
    "order_status",
    "payment_method",
    "total_amount",
    "transaction_timestamp",
    F.explode("items").alias("items")
)
display(exploded_df)


In [0]:
final_df = exploded_df.select(
    "customer_id",
    "order_date",
    "order_id",
    "order_status",
    "payment_method",
    "total_amount",
    "transaction_timestamp",
    "items.item_id",
    "items.name",
    "items.price",
    "items.quantity",
    "items.details.brand",
    "items.details.color"
)
display(final_df)


####4. Write transformed data to silver layer 

In [0]:
final_df.writeTo("gizmobox.silver.py_orders").createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_orders